# Concept 1 practice: self-attention

Reproduces the hand-worked "I love pizza" example in real code, then runs
the scaling investigation we just did together (does `head_dim` change how
sharp the attention distribution gets, with vs without the `sqrt(d_k)`
scaling term).


In [1]:
import numpy as np

def scaled_dot_product_attention(Q, K, V, scale=True):
    """
    Q: (d_k,)          query for one token
    K: (n_tokens, d_k) keys for every token
    V: (n_tokens, d_v) values for every token
    scale: whether to divide by sqrt(d_k), the thing we just derived

    returns (probs, output)
    """
    d_k = Q.shape[-1]
    scores = K @ Q  # dot product of Q against every row of K
    if scale:
        scores = scores / np.sqrt(d_k)
    exp = np.exp(scores - scores.max())  # max-subtract for numerical stability, doesn't change the result
    probs = exp / exp.sum()
    output = probs @ V
    return probs, output


## Reproduce the hand example

"I" / "love" / "pizza", head_dim = 2, computing pizza's new vector.
Hand result was probs = [0.248, 0.248, 0.503], output = [1.759, 1.0].


In [2]:
K = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)  # I, love, pizza
V = np.array([[1, 0], [0, 2], [3, 1]], dtype=float)
Q_pizza = np.array([1, 1], dtype=float)

probs, output = scaled_dot_product_attention(Q_pizza, K, V)
print("probs: ", np.round(probs, 3))
print("output:", np.round(output, 3))

assert np.allclose(probs, [0.248, 0.248, 0.503], atol=1e-3)
assert np.allclose(output, [1.759, 1.0], atol=1e-3)
print("\nmatches the hand calculation.")


probs:  [0.248 0.248 0.503]
output: [1.759 1.   ]

matches the hand calculation.


## The scaling investigation

Same question as in chat: does the attention distribution get sharper as
`head_dim` grows, if we DON'T scale by `sqrt(d_k)`? And does proper scaling
actually neutralize that, like we derived?

A single random draw is too noisy to trust here (one draw of 5 tokens can
easily look sharp or flat by chance). So this averages the entropy of the
softmax output over 5000 random draws, for each combination. Entropy
measures how spread out a distribution is: `log(5) = 1.609` is the max
possible for 5 tokens (perfectly uniform), 0 means fully one-hot.


In [3]:
rng = np.random.default_rng(0)
n_tokens = 5
trials = 5000
max_entropy = np.log(n_tokens)

def entropy(p):
    p = p[p > 1e-12]
    return -(p * np.log(p)).sum()

for d_k in [2, 8, 32, 64]:
    for scale in [False, True]:
        entropies = []
        for _ in range(trials):
            Q = rng.standard_normal(d_k)
            K = rng.standard_normal((n_tokens, d_k))
            V = rng.standard_normal((n_tokens, d_k))
            probs, _ = scaled_dot_product_attention(Q, K, V, scale=scale)
            entropies.append(entropy(probs))
        label = "scaled" if scale else "UNSCALED"
        print(f"d_k={d_k:3d}  {label:9s}  mean entropy = {np.mean(entropies):.3f}  (max possible = {max_entropy:.3f})")


d_k=  2  UNSCALED   mean entropy = 1.204  (max possible = 1.609)
d_k=  2  scaled     mean entropy = 1.347  (max possible = 1.609)


d_k=  8  UNSCALED   mean entropy = 0.738  (max possible = 1.609)


d_k=  8  scaled     mean entropy = 1.316  (max possible = 1.609)


d_k= 32  UNSCALED   mean entropy = 0.350  (max possible = 1.609)


d_k= 32  scaled     mean entropy = 1.311  (max possible = 1.609)


d_k= 64  UNSCALED   mean entropy = 0.248  (max possible = 1.609)


d_k= 64  scaled     mean entropy = 1.307  (max possible = 1.609)


## What to look for

Read down the UNSCALED column first: entropy should fall sharply as `d_k`
grows, from around 1.2 at `d_k=2` down toward 0 by `d_k=64`, that's softmax
saturating into a near one-hot pick, exactly the failure mode scaling
exists to prevent.

Now read the scaled column: it should stay close to flat across every
`d_k`, because scaling is specifically designed to cancel the dimension
effect, this is the actual evidence for the answer you gave in chat.

---

## Your turn

Build a 4-token toy example yourself, any 3 numbers you like for `head_dim`
(try `head_dim = 3`), pick Q/K/V by hand for tokens like "the", "cat",
"sat", "down". Before running it, predict by eye which token the last
token ("down") will attend to most, based on which K vector looks most
similar to "down"'s Q vector. Then run `scaled_dot_product_attention` and
check your prediction. Write your version in the cell below.


In [4]:
# your turn: define K, V (4 tokens x head_dim=3) and Q for "down", then call
# scaled_dot_product_attention(Q, K, V) and check your prediction

